## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 40)

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

# Path to raw data
RAW_DATA_PATH = Path('../data/raw/raw_data.xlsx')
TARGET = 'Churned'

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'Data path exists: {RAW_DATA_PATH.exists()}')

---
## 2. Load Raw Excel Sheets

The dataset consists of **3 relational sheets** inside `raw_data.xlsx`:

| Sheet | Role | Join Key |
|---|---|---|
| `Demographic` | Customer profile + target | `CustomerId`, `LocationId` |
| `Location` | Geography lookup table | `LocationId` |
| `Account` | Financial & product data | `CustomerId` |

In [ ]:
# Load all 3 sheets
df_demo = pd.read_excel(RAW_DATA_PATH, sheet_name='Demographic')
df_loc  = pd.read_excel(RAW_DATA_PATH, sheet_name='Location')
df_acc  = pd.read_excel(RAW_DATA_PATH, sheet_name='Account')

print('Sheet shapes loaded:')
print(f'  Demographic : {df_demo.shape}')
print(f'  Location    : {df_loc.shape}')
print(f'  Account     : {df_acc.shape}')

In [ ]:
# Verify all sheet names present in the file
xl = pd.ExcelFile(RAW_DATA_PATH)
print('Sheets in raw_data.xlsx:', xl.sheet_names)

### Demographic

In [ ]:
df_demo.head()

In [ ]:
print('=== Demographic Sheet ===')
print(f'Shape       : {df_demo.shape}')
print(f'Columns     : {list(df_demo.columns)}')
print()
print('--- Data Types ---')
print(df_demo.dtypes)
print()
print('--- Missing Values ---')
missing = df_demo.isnull().sum()
pct     = (missing / len(df_demo) * 100).round(2)
print(pd.DataFrame({'Missing': missing, 'Pct (%)': pct}))
print()
print(f'Duplicate rows: {df_demo.duplicated().sum()}')

In [ ]:
# Statistical summary — numerical columns
df_demo.describe()

In [ ]:
# Categorical columns in Demographic
print('--- Categorical Column Cardinality ---')
for col in df_demo.select_dtypes('object').columns:
    print(f'\n{col} — {df_demo[col].nunique()} unique values')
    print(df_demo[col].value_counts().head(10).to_string())

In [ ]:
# Data quality checks — Phase 2.2 from e2e guide
print('=== Data Quality Checks — Demographic ===')

# 1. String NaN variants
str_cols = df_demo.select_dtypes('object').columns
for col in str_cols:
    nan_variants = df_demo[col].isin(['NaN', 'nan', 'NA', 'null', 'NULL', '', ' ']).sum()
    print(f'[String NaN] {col}: {nan_variants} occurrences')

# 2. Age validity (must be 18–100)
invalid_age = df_demo[(df_demo['Age'] < 18) | (df_demo['Age'] > 100)].shape[0]
print(f'\n[Age] Values outside [18, 100]: {invalid_age}')
print(f'[Age] Min: {df_demo["Age"].min()}, Max: {df_demo["Age"].max()}, Mean: {df_demo["Age"].mean():.1f}')

# 3. Salary validity
invalid_salary = df_demo[df_demo['Salary'] <= 0].shape[0]
print(f'\n[Salary] Values <= 0: {invalid_salary}')
print(f'[Salary] Min: {df_demo["Salary"].min():.2f}, Max: {df_demo["Salary"].max():.2f}')

# 4. LocationId valid range (1–6, matching Location sheet)
print(f'\n[LocationId] Unique values: {sorted(df_demo["LocationId"].unique())}')

# 5. Churned must be binary
print(f'\n[Churned] Unique values: {sorted(df_demo["Churned"].unique())}')

#### Location

In [ ]:
# Small lookup table — show completely
print(f'Shape: {df_loc.shape}')
print()
display(df_loc)
print()
print(f'Dtypes:\n{df_loc.dtypes}')
print(f'\nMissing values: {df_loc.isnull().sum().sum()}')
print(f'Duplicate rows: {df_loc.duplicated().sum()}')

In [ ]:
# How many customers per Geography?
geo_counts = (
    df_demo
    .merge(df_loc, on='LocationId', how='left')
    ['Geography']
    .value_counts()
    .reset_index()
)
geo_counts.columns = ['Geography', 'CustomerCount']
geo_counts['Pct (%)'] = (geo_counts['CustomerCount'] / len(df_demo) * 100).round(2)
print('Customer distribution by Geography:')
display(geo_counts)

### Account

In [ ]:
df_acc.head()

In [ ]:
print('=== Account Sheet ===')
print(f'Shape       : {df_acc.shape}')
print(f'Columns     : {list(df_acc.columns)}')
print()
print('--- Data Types ---')
print(df_acc.dtypes)
print()
print('--- Missing Values ---')
missing_acc = df_acc.isnull().sum()
pct_acc     = (missing_acc / len(df_acc) * 100).round(2)
print(pd.DataFrame({'Missing': missing_acc, 'Pct (%)': pct_acc}))
print()
print(f'Duplicate rows: {df_acc.duplicated().sum()}')

In [ ]:
df_acc.describe()

In [ ]:
print('=== Data Quality Checks — Account ===')

# NumProducts valid range (1–4)
print(f'[NumProducts] Unique values: {sorted(df_acc["NumProducts"].unique())}')
invalid_products = df_acc[~df_acc['NumProducts'].isin([1, 2, 3, 4])].shape[0]
print(f'[NumProducts] Values outside [1, 4]: {invalid_products}')

# HasCreditCard must be binary
print(f'\n[HasCreditCard] Unique values: {sorted(df_acc["HasCreditCard"].unique())}')

# IsActive must be binary
print(f'[IsActive] Unique values: {sorted(df_acc["IsActive"].unique())}')

# Tenure range (0–10)
print(f'\n[Tenure] Min: {df_acc["Tenure"].min()}, Max: {df_acc["Tenure"].max()}')

# Balance — missing values
balance_missing = df_acc['Balance'].isnull().sum()
balance_pct     = balance_missing / len(df_acc) * 100
print(f'\n[Balance] Missing: {balance_missing} ({balance_pct:.2f}%) ⚠️ HIGH MISSINGNESS')
print(f'[Balance] Min: {df_acc["Balance"].min():.2f}, Max: {df_acc["Balance"].max():.2f}')
print(f'[Balance] Median: {df_acc["Balance"].median():.2f}')

# AccountId uniqueness
print(f'\n[AccountId] Unique: {df_acc["AccountId"].nunique()} / {len(df_acc)} (100% unique = identifier ✅)')

In [ ]:
print('=== Relational Integrity Checks ===')

# Check 1: All LocationIds in Demographic exist in Location
demo_loc_ids = set(df_demo['LocationId'].unique())
loc_ids      = set(df_loc['LocationId'].unique())
orphaned_loc = demo_loc_ids - loc_ids
print(f'Demographic LocationIds    : {sorted(demo_loc_ids)}')
print(f'Location LocationIds       : {sorted(loc_ids)}')
print(f'Orphaned LocationIds       : {orphaned_loc} (should be empty set)')

print()

# Check 2: All CustomerId in Demographic exist in Account
demo_cids = set(df_demo['CustomerId'].unique())
acc_cids  = set(df_acc['CustomerId'].unique())
orphaned_demo = demo_cids - acc_cids
orphaned_acc  = acc_cids - demo_cids
print(f'Demographic CustomerId count : {len(demo_cids)}')
print(f'Account CustomerId count     : {len(acc_cids)}')
print(f'In Demographic NOT in Account: {len(orphaned_demo)} (should be 0)')
print(f'In Account NOT in Demographic: {len(orphaned_acc)} (should be 0)')

In [ ]:
# Execute the relational join
df_merged = (
    df_demo
    .merge(df_loc, on='LocationId', how='left')   # Bring in Geography
    .merge(df_acc, on='CustomerId', how='left')   # Bring in Account info
)

print(f'Post-join shape: {df_merged.shape}')
print(f'Expected       : (10000, 14)')
assert df_merged.shape[0] == len(df_demo), 'Row count changed after merge!'
print('\nJoin integrity: ✅ Row count preserved')
df_merged.head(3)

### Merged data view

In [ ]:
print('=== Combined Dataset (Post-Join) ===')
print(f'Shape   : {df_merged.shape}')
print(f'Columns : {list(df_merged.columns)}')
print()
print('--- Data Types ---')
print(df_merged.dtypes)
print()
print('--- Missing Values (All Columns) ---')
missing_m = df_merged.isnull().sum()
pct_m     = (missing_m / len(df_merged) * 100).round(2)
mv_df     = pd.DataFrame({'Missing': missing_m, 'Pct (%)': pct_m})
print(mv_df[mv_df['Missing'] > 0])
print(f'\nColumns with zero missing: {(mv_df["Missing"] == 0).sum()} / {len(mv_df)}')

In [ ]:
df_merged.describe(include='all')

### Target

In [ ]:
churn_counts = df_merged[TARGET].value_counts()
churn_pct    = df_merged[TARGET].value_counts(normalize=True) * 100

print('=== Target: Churned ===')
print(f'0 = Stayed : {churn_counts[0]:,} ({churn_pct[0]:.2f}%)')
print(f'1 = Churned: {churn_counts[1]:,} ({churn_pct[1]:.2f}%)')
print(f'\nClass imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1  ⚠️ Imbalanced')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(['Stayed (0)', 'Churned (1)'], churn_counts.values,
                   color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Target Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(churn_counts.values,
            labels=[f'Stayed\n{churn_pct[0]:.1f}%', f'Churned\n{churn_pct[1]:.1f}%'],
            colors=['#2ecc71', '#e74c3c'],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target Distribution (%)', fontweight='bold')

plt.suptitle('Class Imbalance Analysis — Churned Target', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n⚠️ Imbalanced dataset: requires class_weight="balanced" or scale_pos_weight in models')